In [3]:
library(rjags)
library(coda)

In [4]:
PATH_DATA_CLEAN = "../data/clean/"
PATH_DATA_OUT = "../data/"
PATH_CODE = "./"

# Production hyperparameters (runtime: ~6-10h on a laptop)
ADAPT  = 1000
BURNIN = 5000
DRAWS  = 25000
THIN   = 10
CHAINS = 2

In [5]:
# # Test hyperparameters — runs in ~1-5 min; swap for production values above when ready
# ADAPT  = 100
# BURNIN = 200
# DRAWS  = 500
# THIN   = 2
# CHAINS = 1

In [6]:
# 1. Load cleaned outputs from clean.ipynb
panel_df = read.csv(paste0(PATH_DATA_CLEAN, "merged_panel.csv"))
y_dt = as.matrix(read.csv(paste0(PATH_DATA_CLEAN, "y_matrix.csv")))
indices_df = read.csv(paste0(PATH_DATA_CLEAN, "panel_indices.csv"))
lens_df = read.csv(paste0(PATH_DATA_CLEAN, "panel_lengths.csv"))

country_idx = indices_df$country
year_idx = indices_df$year
panel_lens = lens_df$panel_length

cat(sprintf("Loaded: %d rows, %d countries, %d indicators\n",
            nrow(y_dt), length(panel_lens), ncol(y_dt)))

Loaded: 9268 rows, 204 countries, 13 indicators


In [7]:
# 2. Initial values generator — static model
# Alpha arrays are 2D [item, cut] because cut points are fixed over time

make_inits_static = function() {
  n_binary = 5
  n_ord3 = 4
  n_ord5 = 3
  n_ord6 = 1
  n_countries = length(panel_lens)

  MU = matrix(rnorm(n_countries * max(panel_lens), mean = 0, sd = 1),
              nrow = n_countries, ncol = max(panel_lens))

  # NOTE: drawn from successive intervals to satisfy JAGS's ordering constraint
  ALPHA03 = matrix(c(runif(n_ord3, 0, 1), runif(n_ord3, 1, 2)),
                   nrow = n_ord3, ncol = 2)
  ALPHA05 = matrix(c(runif(n_ord5, 0.0, 0.5), runif(n_ord5, 0.5, 1.0),
                     runif(n_ord5, 1.0, 1.5), runif(n_ord5, 1.5, 2.0)),
                   nrow = n_ord5, ncol = 4)
  ALPHA06 = matrix(c(runif(n_ord6, 0.0, 0.5), runif(n_ord6, 0.5, 1.0),
                     runif(n_ord6, 1.0, 1.5), runif(n_ord6, 1.5, 2.0),
                     runif(n_ord6, 2.0, 2.5)),
                   nrow = n_ord6, ncol = 5)

  list(
    mu = MU,
    alpha1 = runif(n_binary),
    beta1 = runif(n_binary),
    alpha03 = ALPHA03,
    beta3 = runif(n_ord3),
    alpha05 = ALPHA05,
    beta5 = runif(n_ord5),
    alpha06 = ALPHA06,
    beta6 = runif(n_ord6),
    sigma = runif(1)
  )
}

cat("Init function ready\n")

Init function ready


In [8]:
# 3. Compile and run burn-in — static model
# Baseline model: cut points are fixed over time — no time index passed to JAGS

MODEL_STATIC = paste0(PATH_CODE, "LatentRepressionConstantStandardDynamicX.bug")

# NOTE: time_idx excluded — the static model doesn't use it and JAGS warns if passed
jags_data_static = list(
  y = y_dt,
  year = year_idx,
  country = country_idx,
  n.country = length(panel_lens),
  n.year = max(panel_lens),
  n = nrow(y_dt)
)

inits_static = lapply(seq_len(CHAINS), function(i) make_inits_static())
inits_fn_static = function(chain) inits_static[[chain]]

t_start = Sys.time()
cat("Compiling static model...\n")

m_static = jags.model(
  file = MODEL_STATIC,
  data = jags_data_static,
  inits = inits_fn_static,
  n.chains = CHAINS,
  n.adapt = ADAPT
)

cat(sprintf("Adaptation done (%.1f min)\n", as.numeric(Sys.time() - t_start, units = "mins")))
update(m_static, BURNIN)
cat(sprintf("Burn-in done (%.1f min)\n", as.numeric(Sys.time() - t_start, units = "mins")))

Compiling static model...
Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 59698
   Unobserved stochastic nodes: 73478
   Total graph size: 1134171

Initializing model

Adaptation done (105.6 min)
Burn-in done (416.8 min)


In [ ]:
# 4. Draw posterior samples and save
samples_static = coda.samples(
  m_static,
  variable.names = c("x", "beta1", "beta3", "beta5", "beta6",
                     "alpha1", "alpha3", "alpha5", "alpha6",
                     "kappa", "sigma"),
  n.iter = DRAWS,
  thin = THIN,
  progress.bar = "text"
)

cat(sprintf("Sampling done (%.1f min)\n", as.numeric(Sys.time() - t_start, units = "mins")))

posterior_static = do.call(rbind, lapply(samples_static, as.matrix))
write.csv(as.data.frame(posterior_static),
          paste0(PATH_DATA_OUT, "EstimateConstantStandardDynamicX.csv"),
          row.names = FALSE)

save.image(paste0(PATH_DATA_OUT, "image_static.Rdata"))
cat("Static model saved\n")

In [ ]:
# 5. Convergence diagnostics
# Gelman-Rubin R-hat: values close to 1.0 indicate convergence
# NOTE: only meaningful with CHAINS >= 2; skipped in test mode

if (CHAINS >= 2) {
  gr_static = gelman.diag(samples_static, multivariate = FALSE)
  cat(sprintf("Static — max R-hat: %.3f | params > 1.1: %d\n",
              max(gr_static$psrf[, 1], na.rm = TRUE),
              sum(gr_static$psrf[, 1] > 1.1, na.rm = TRUE)))
} else {
  cat("Skipping Gelman-Rubin: set CHAINS = 2 in production\n")
}

Skipping Gelman-Rubin: set CHAINS = 2 in production
